# 13 — Familia F8: supervisados (D16 logit L1 · D17 random forest)

Dos detectores **supervisados** de la Fase 2 ([ADR-010](../docs/decisions/ADR-010-preregistro-pool-crudo.md) §3):

| id | modelo | hiperparámetros (fijos, sin afinar con crisis) |
|---|---|---|
| D16 | logit con penalización L1 | `C = 1`, `class_weight = "balanced"`, `liblinear`; crisis si p ≥ 0,5 |
| D17 | random forest | 500 árboles, `max_depth = 5`, `min_samples_leaf = 50`, `balanced_subsample`, semilla 42; crisis si p ≥ 0,5 |

Cada uno en dos menús: **P** (pool completo, la variante principal, con la regla de disponibilidad del §2.1) y **N**
(núcleo, la comparación barata; antes era T, descartado en el ADR-009). Mismo walk-forward que v2, paso 21.

**Etiquetas sin fuga (§3.2).** En cada reajuste solo existen como etiqueta las crisis cuyo suelo ya ha pasado; los
días de una crisis aún sin terminar cuentan como calma, igual que los vería alguien en tiempo real. Las trampas
cuentan como calma. La sensibilidad `-excl` saca esos días del ajuste; no pesa en ningún veredicto.

**Categoría aparte (§6, «techo»).** Usan las etiquetas que el resto solo ve para puntuar, así que no entran en el
ranking principal ni en el criterio del §5. Responden otra pregunta: **¿hay en los datos información que los no
supervisados no extraen?**

**Índice**

1. Estado de los resultados
2. Etiquetas en cada reajuste
3. Detección: D16 y D17 en P y en N
4. Techo: frente al mejor no supervisado (§6)
5. Sensibilidad `-excl`
6. Coeficientes e importancias (ilustrativo)
7. Lectura

Solo lee `results/detectores/pool/` (lo escriben `python -m regimenes.benchmark.pool ejecutar` y `leer`). Si falta
algo, cada celda avisa y no falla. Siguiente: `14_comparativa` §13.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3, 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.width', 200)

from regimenes import rutas
from regimenes.benchmark import pool_lectura as pl

PISTAS = ('A', 'B')
IDS_SUP = ['D16-P', 'D16-N', 'D17-P', 'D17-N', 'D16-P-excl', 'D17-P-excl']
print('Resultados en', pl.rel(pl.SALIDA))

## 1. Estado de los resultados

In [ ]:
display(pl.estado())
VAR = pl.variantes()
SUP = VAR[VAR['id'].isin(IDS_SUP)].copy() if len(VAR) else VAR
HAY_SUP = len(SUP) > 0
faltan = [f'{t}/{i}' for t in PISTAS for i in IDS_SUP if not ((SUP['pista'] == t) & (SUP['id'] == i)).any()] \
    if HAY_SUP else [f'{t}/{i}' for t in PISTAS for i in IDS_SUP]
print(f'Supervisados terminados: {len(SUP)} de {2 * len(IDS_SUP)}.',
      f'AVISO: faltan {faltan}; las tablas son PROVISIONALES.' if faltan else '')

## 2. Etiquetas en cada reajuste

Crisis terminadas y días de crisis que ve cada reajuste. Mientras no hay ninguna crisis terminada el detector
predice calma (`sup_siempre_calma`).

In [ ]:
filas = []
for t in PISTAS:
    for i in ('D16-P', 'D16-N', 'D16-P-excl'):
        r = pl.reajustes(t, i)
        if r.empty or 'sup_n_crisis_terminadas' not in r:
            continue
        calma = r['sup_siempre_calma'].eq(True)
        filas.append({'pista': t, 'id': i, 'reajustes': len(r), 'siempre_calma': int(calma.sum()),
                      'crisis_terminadas_inicio': int(r['sup_n_crisis_terminadas'].iloc[0]),
                      'crisis_terminadas_final': int(r['sup_n_crisis_terminadas'].iloc[-1]),
                      'dias_crisis_final': int(r['sup_n_dias_crisis'].iloc[-1]),
                      'filas_ajuste_final': int(r['sup_n_filas_ajuste'].iloc[-1])})
if filas:
    display(pd.DataFrame(filas).set_index(['pista', 'id']))
    print('Las etiquetas son las mismas en D16 y D17 (mismo menú y misma regla); se muestra D16.')
else:
    print(f'{pl.PENDIENTE}: faltan los reajustes de D16 (reajustes/<P>_D16-*.csv).')

## 3. Detección: D16 y D17 en P y en N

`score_deteccion` (ADR-003) con recall por evento y precisión. `delta_P_menos_N` dice si el pool completo ayuda a
cada modelo frente al núcleo.

In [ ]:
COLS = ['menu', 'score_deteccion', 'det_event_recall', 'det_precision', 'det_lift_precision',
        'det_marked_rate', 'mean_regime_duration', 'n_reajustes']
PRINCIPALES = SUP[~SUP['id'].str.endswith('-excl')] if HAY_SUP else SUP
if len(PRINCIPALES):
    display(PRINCIPALES.set_index(['pista', 'id'])[[c for c in COLS if c in PRINCIPALES]].round(3))
    s = PRINCIPALES.set_index(['pista', 'base', 'menu'])['score_deteccion'].unstack('menu')
    if {'P', 'N'} <= set(s.columns):
        s['delta_P_menos_N'] = s['P'] - s['N']
    display(s.round(3))
else:
    print(f'{pl.PENDIENTE}: D16 y D17 en P y N.')

## 4. Techo: frente al mejor no supervisado (§6)

Cada supervisado frente al mejor no supervisado de su pista (los 12 de v2 y las variantes no supervisadas del pool
terminadas), con `score_deteccion`, y su presencia en un MCS de supervisados y no supervisados juntos, solo como
lectura. Si ninguno mejora al mejor no supervisado, ni con etiquetas se gana a lo simple; si alguno lo mejora, hay
información que los no supervisados no extraen. En ningún caso cambia el ranking principal.

In [ ]:
TECHO = pl.leer_tabla('techo_supervisados')
aviso = pl.aviso_tabla('techo_supervisados.csv', TECHO)
if TECHO is None or TECHO.attrs.get('desfasada'):
    print(aviso, 'Se muestra el score sin MCS, calculado con lo terminado.')
    TECHO = pl.techo()
if len(TECHO):
    display(TECHO.set_index(['pista', 'id']).round(3))
    print('Mejoran al mejor no supervisado:', list(TECHO.loc[TECHO['mejora_al_mejor'], 'id']
                                                   + '/' + TECHO.loc[TECHO['mejora_al_mejor'], 'pista']) or 'ninguno')
    if 'en_mcs_conjunto' in TECHO:
        print('En el MCS conjunto:', [f'{r.pista}/{r.id}' for r in TECHO.itertuples() if r.en_mcs_conjunto] or 'ninguno')
else:
    print(f'{pl.PENDIENTE}: techo de los supervisados.')
for t in PISTAS:
    MCS_SUP = pl.leer_tabla(f'mcs_supervisados_{t}', index_col='modelo')
    if MCS_SUP is None:
        print(pl.aviso_tabla(f'mcs_supervisados_{t}.csv', MCS_SUP))
        continue
    dentro = [str(i) for i in MCS_SUP.index[MCS_SUP['incluido'].astype(bool)]]
    print(f'Pista {t} · MCS conjunto al 10 % (bloque {MCS_SUP["bloque"].iloc[0]}): mejor {MCS_SUP["perdida_media"].idxmin()}; '
          f'dentro {dentro}', pl.aviso_tabla('', MCS_SUP) or '')

## 5. Sensibilidad `-excl`

La variante que saca del ajuste los días de crisis aún sin terminar, frente a la principal en P. Sin peso en el
veredicto: elige las filas con una etiqueta futura (§3.2).

In [ ]:
if HAY_SUP:
    s = SUP.set_index(['pista', 'id'])['score_deteccion']
    filas = [{'pista': t, 'modelo': b, 'principal': s.get((t, f'{b}-P')), 'excl': s.get((t, f'{b}-P-excl'))}
             for t in PISTAS for b in ('D16', 'D17')]
    EXCL = pd.DataFrame(filas).dropna(subset=['principal', 'excl'], how='all')
    EXCL['delta_excl_menos_principal'] = EXCL['excl'] - EXCL['principal']
    if len(EXCL):
        display(EXCL.set_index(['pista', 'modelo']).round(3))
    else:
        print(f'{pl.PENDIENTE}: sensibilidad -excl (faltan D16-P/D17-P y sus -excl).')
else:
    print(f'{pl.PENDIENTE}: sensibilidad -excl.')

## 6. Coeficientes e importancias (ilustrativo)

Del ajuste final con toda la muestra (todas las crisis terminadas), el mismo que hace el ejecutor al terminar; no
son los modelos de cada reajuste. D16: coeficientes del logit sobre z-scores (L1 deja muchos a 0). D17: importancia
por reducción de impureza. Se calcula solo para las variantes terminadas; el random forest tarda unos segundos.

In [ ]:
CALCULAR_IMPORTANCIAS = True
IMP = {}
if CALCULAR_IMPORTANCIAS and HAY_SUP:
    for t in PISTAS:
        for i in ('D16-P', 'D17-P', 'D16-N', 'D17-N'):
            if ((SUP['pista'] == t) & (SUP['id'] == i)).any():
                imp = pl.importancias(t, i)
                if imp is not None:
                    IMP[t, i] = imp
if IMP:
    for (t, i), v in IMP.items():
        if i.startswith('D16'):
            print(f'{t}/{i}: {int((v != 0).sum())} de {len(v)} coeficientes distintos de 0.')
    principales = [(t, i) for (t, i) in IMP if i.endswith('-P')]
    if principales:
        fig, axes = plt.subplots(1, len(principales), figsize=(4.2 * len(principales), 4.2), squeeze=False)
        for ax, (t, i) in zip(axes[0], principales):
            v = IMP[t, i]
            top = v.reindex(v.abs().sort_values().index[-15:])
            ax.barh(top.index, top.to_numpy(), color=np.where(top.to_numpy() >= 0, 'C3', 'C0'))
            ax.set_title(f'{t} · {i}: ' + ('coeficientes (15 mayores |β|)' if i.startswith('D16') else 'importancias (15 mayores)'),
                         fontsize=8)
            ax.tick_params(axis='y', labelsize=7)
        plt.tight_layout()
        plt.show()
    N_TAB = {f'{t}/{i}': v for (t, i), v in IMP.items() if i.endswith('-N')}
    if N_TAB:
        display(pd.DataFrame(N_TAB).round(3))
else:
    print(f'{pl.PENDIENTE}: coeficientes e importancias (faltan las variantes o la entrada de pool preparar).')

## 7. Lectura

**PENDIENTE DE RESULTADOS.** Se rellena cuando terminen las variantes y `pool leer` (sesión 2 del ADR-010 §9):

- **D16 y D17 en P frente a N:** ¿saca partido del pool un modelo con etiquetas? PENDIENTE DE RESULTADOS.
- **Techo (§6):** ¿mejora alguno al mejor no supervisado o entra en el MCS conjunto? PENDIENTE DE RESULTADOS.
- **Sensibilidad `-excl`:** PENDIENTE DE RESULTADOS.
- **Qué columnas usan:** PENDIENTE DE RESULTADOS.